# Курсовая работа — распознавание и синонимическое обезличивание ПДн

10 классов, RuBERT + правила. Обучение отключено: блок 4 только загружает уже сохранённую модель.

**Важное ограничение:** предыдущий контрольный набор уже использовался для анализа ошибок и теперь считается регрессионным, а не независимым. Для подтверждения F1 требуется ещё один новый набор, не участвующий в корректировках. Метрики нового запуска не предзаполнены.


In [1]:
#===== Блок 1. Минимальные зависимости для оценки без обучения =====
%pip install -q "pymorphy3>=2" "bert-score>=0.3.13" "transformers>=4.50,<5"


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
"""КР: распознавание и синонимическое обезличивание 10 категорий ПДн.
Python 3.8+. Для RuBERT передайте функцию ner(text) -> [TYPE, start, end].
"""
import csv
import json
import random
import re
import secrets
from collections import Counter
from pathlib import Path

TYPES = ('SURNAME', 'NAME', 'INN', 'PASSPORT', 'CARD', 'SNILS',
         'URL', 'PHONE', 'MARRIAGE', 'CADASTRAL')
TOKEN = re.compile(r'\w+|[^\w\s]', re.UNICODE)
PATTERNS = {
    'URL': re.compile(r'(?i)(?<![@\w])https?://[^\s<>"“”]+|(?<![@\w])(?:[a-z0-9-]+\.)+(?:ru|рф|com|org|net|edu)(?:/[^\s<>"“”]*)?', re.I),
    'INN': re.compile(r'(?<!\d)(?:\d[ -]?){9,11}\d(?!\d)'),
    'CARD': re.compile(r'(?<!\d)(?:\d[ -]?){15,18}\d(?!\d)'),
    'SNILS': re.compile(r'(?<!\d)\d{3}[- ]?\d{3}[- ]?\d{3}[ -]?\d{2}(?!\d)'),
    'PHONE': re.compile(r'(?<!\w)(?:\+7|8)\s*\(?\d{3}\)?[ -]?\d{3}[ -]?\d{2}[ -]?\d{2}(?!\w)'),
    'PASSPORT': re.compile(r'(?<!\w)\d{2}[ -]?\d{2}[ -]?\d{6}(?!\w)'),
    'MARRIAGE': re.compile(r'(?<!\w)(?:I|II|III|IV|V|VI|VII|VIII|IX|X)[-– ]?[А-ЯЁ]{2,3}\s*(?:№\s*)?\d{6}(?!\w)', re.I),
    'CADASTRAL': re.compile(r'(?<![\d:])\d{2}:\d{2}:\d{6,7}:\d{1,12}(?![\d:])'),
}
PRIORITY = ('URL', 'CADASTRAL', 'MARRIAGE', 'SNILS', 'CARD', 'PHONE', 'PASSPORT', 'INN')
CONTEXT = {
    'INN': (r'(?:\bИНН\b|идентификационн(?:ый|ого)\s+номер\s+налогоплательщика|'
            r'налогов(?:ый|ого)\s+идентификатор(?:\s+заявителя)?|'
            r'налогоплательщик\s+сообщил\s+ИНН|реквизиты\s+организации\s*:\s*ИНН|'
            r'налогов(?:ый|ого)\s+номер|идентификатор\s+налогоплательщика)'
            r'\s*(?:№|:|—|–|-)?\s*$'),
    'PASSPORT': (r'(?:паспорт[а-я]*(?:\s+РФ)?|серия\s+паспорта|'
                 r'паспортн(?:ые|ых)\s+(?:сведения|данные)|паспортн(?:ый|ого)\s+номер|'
                 r'удостоверяющ(?:ий|его)\s+документ|документ\s+удостоверения\s+личности)'
                 r'(?:\s+\w+){0,3}\s*(?:номер|№|:|—|–|-)?\s*$'),
    'MARRIAGE': (r'(?:свидетельств[оа]\s+о\s+(?:заключении\s+)?брак[ае]|'
                 r'акт(?:овая\s+запись|а|е)?\s+о\s+браке|'
                 r'регистрации\s+брака|брачн(?:ое|ого)\s+свидетельств[оа]|'
                 r'документ\s+о\s+заключении\s+брака|'
                 r'реквизиты\s+акта\s+о\s+браке)'
                 r'\s*(?:(?:серия|серии)|номер|№|:|—|–|-)?\s*$'),
}

MALE_NAMES = ('Иван', 'Пётр', 'Алексей', 'Дмитрий', 'Михаил', 'Андрей', 'Сергей', 'Николай')
FEMALE_NAMES = ('Анна', 'Мария', 'Елена', 'Ольга', 'Ирина', 'Наталья', 'Татьяна', 'Светлана')
MALE_SURNAMES = ('Иванов', 'Петров', 'Сидоров', 'Смирнов', 'Кузнецов', 'Орлов', 'Волков', 'Фёдоров')
FEMALE_SURNAMES = tuple(n + 'а' for n in MALE_SURNAMES)


#===== Проверка контрольного числа ИНН =====
def valid_inn(value):
    s = re.sub(r'\D', '', value)
    if len(s) == 10:
        return sum(int(a)*b for a, b in zip(s[:9], (2, 4, 10, 3, 5, 9, 4, 6, 8))) % 11 % 10 == int(s[-1])
    if len(s) == 12:
        p = sum(int(a)*b for a, b in zip(s[:10], (7, 2, 4, 10, 3, 5, 9, 4, 6, 8))) % 11 % 10
        q = sum(int(a)*b for a, b in zip(s[:11], (3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8))) % 11 % 10
        return (p, q) == (int(s[-2]), int(s[-1]))
    return False


#===== Проверка номера карты по Луну =====
def valid_card(value):
    s = re.sub(r'[ -]', '', value)
    if not s.isdigit() or not 16 <= len(s) <= 19:
        return False
    digits = [int(c) for c in s[::-1]]
    return sum((x * 2 - 9 if x * 2 > 9 else x * 2) if i % 2 else x
               for i, x in enumerate(digits)) % 10 == 0


#===== Проверка контрольного числа СНИЛС =====
def valid_snils(value):
    s = re.sub(r'\D', '', value)
    if len(s) != 11 or int(s[:9]) <= 1001998:
        return False
    number = sum(int(v) * (9 - i) for i, v in enumerate(s[:9]))
    checksum = number if number < 100 else (0 if number in (100, 101) else number % 101)
    if checksum == 100:
        checksum = 0
    return checksum == int(s[-2:])


#===== Проверка структурированного атрибута =====
def valid_value(kind, value):
    if kind == 'INN':
        return valid_inn(value)
    if kind == 'CARD':
        return valid_card(value)
    if kind == 'SNILS':
        return valid_snils(value)
    if kind == 'CADASTRAL':
        a, b, c, d = map(int, value.split(':'))
        return 1 <= a <= 99 and 0 <= b <= 99 and c > 0 and d > 0
    return True


#===== Выделение структурированных атрибутов с контекстом =====
def find_structured(text):
    found = []
    for kind in PRIORITY:
        for match in PATTERNS[kind].finditer(text):
            start, stop = match.span()
            value = match.group().rstrip('.,;!?') if kind == 'URL' else match.group()
            stop = start + len(value)
            if kind in CONTEXT and not re.search(CONTEXT[kind], text[max(0, start-110):start], re.I):
                continue
            if not valid_value(kind, value):
                continue
            if not any(start < end and begin < stop for _, begin, end in found):
                found.append([kind, start, stop])
    return sorted(found, key=lambda e: e[1])


#===== Распознавание подписанных полей ФИО =====
def find_person_fields(text):
    prefixes = {
        'NAME': (r'\b(?:имя|имени)(?:\s+\w+){0,2}\s*[:–—-]\s*',
                 r'\b(?:по\s+имени|назвал(?:а)?\s+себя|представил(?:ся|ась))\s*[:–—-]?\s*'),
        'SURNAME': (r'\b(?:фамилия|фамилию)(?:\s+\w+){0,2}\s*[:–—-]\s*',
                    r'\b(?:ответственный\s+за\s+документ|в\s+списке\s+участников\s+значится)\s*[:–—-]?\s*',
                    r'\b(?:с\s+фамилией|по\s+фамилии|указана\s+фамилия|'
                    r'подпись\s+заявителя)\s*[:–—-]?\s*'),
    }
    found = []
    for kind, patterns in prefixes.items():
        for prefix in patterns:
            for match in re.finditer('(?i:' + prefix + r')([А-ЯЁ][а-яё]+)\b', text):
                found.append([kind, match.start(1), match.end(1)])
    return found


#===== Распознавание ФИО в обращениях и документных фразах =====
def find_person_context(text):
    found = []
    pair = (r'\b(?:гражданин|гражданка|сотрудник|сотрудница|заявитель|заявительница)'
            r'\s+([А-ЯЁ][а-яё]+)\s+([А-ЯЁ][а-яё]+)\b')
    for match in re.finditer('(?i:' + pair + ')', text):
        found.extend((['SURNAME', match.start(1), match.end(1)],
                      ['NAME', match.start(2), match.end(2)]))
    return found


#===== Единое распознавание без пересечений =====
def recognize_pii(text, ner=None):
    found = find_structured(text)
    for kind, start, stop in find_person_fields(text) + find_person_context(text):
        if not any(start < e and s < stop for _, s, e in found):
            found.append([kind, start, stop])
    if ner is not None:
        for kind, start, stop in ner(text):
            if kind not in ('NAME', 'SURNAME') or not 0 <= start < stop <= len(text):
                continue
            # Не принимаем строчные общеупотребительные слова за ФИО.
            if not re.fullmatch(r'[А-ЯЁ][а-яё]+(?:-[А-ЯЁ][а-яё]+)?', text[start:stop]):
                continue
            if not any(start < e and s < stop for _, s, e in found):
                found.append([kind, start, stop])
    return sorted(found, key=lambda e: e[1])

#===== Преобразование спанов в токенную BIO-разметку =====
def spans_to_bio(text, spans):
    words = list(TOKEN.finditer(text))
    tags = ['O'] * len(words)
    for kind, start, stop in sorted(spans, key=lambda x: x[1]):
        ids = [i for i, word in enumerate(words) if word.start() < stop and start < word.end()]
        if kind not in TYPES or not ids or any(tags[i] != 'O' for i in ids):
            continue
        for j, i in enumerate(ids):
            tags[i] = ('B-' if j == 0 else 'I-') + kind
    return tags


#===== Выделение сущностей по BIO-тегам =====
def bio_entities(tags):
    out, active = set(), None
    for i, tag in enumerate(list(tags) + ['O']):
        prefix, kind = tag.split('-', 1) if '-' in tag else ('O', '')
        if active and (prefix != 'I' or kind != active[0]):
            out.add((active[0], active[1], i))
            active = None
        if prefix in ('B', 'I') and active is None:
            active = (kind, i)
    return out


#===== Метрики классификации токенов по сущностям BIO =====
def evaluate(cases, ner=None, directory='kr_results'):
    result = {k: Counter() for k in TYPES}
    errors = []
    counts = Counter()
    for case in cases:
        text, expected = case['text'], case['spans']
        predicted = recognize_pii(text, ner)
        gold = bio_entities(spans_to_bio(text, expected))
        guess = bio_entities(spans_to_bio(text, predicted))
        for kind in TYPES:
            g = {e for e in gold if e[0] == kind}
            p = {e for e in guess if e[0] == kind}
            result[kind]['tp'] += len(g & p)
            result[kind]['fp'] += len(p - g)
            result[kind]['fn'] += len(g - p)
            counts[kind] += len(g)
        if gold != guess:
            errors.append({'text': text, 'gold': sorted(gold), 'predicted': sorted(guess)})
    rows = []
    for kind, c in result.items():
        tp, fp, fn = (c[key] for key in ('tp', 'fp', 'fn'))
        precision = tp / (tp + fp) if tp + fp else 0
        recall = tp / (tp + fn) if tp + fn else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
        rows.append(dict(attribute=kind, precision=precision, recall=recall,
                         f1=f1, support=tp+fn, tp=tp, fp=fp, fn=fn))
    macro = sum(r['f1'] for r in rows) / len(rows)
    support = sum(r['support'] for r in rows)
    weighted = sum(r['f1'] * r['support'] for r in rows) / support if support else 0
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    with (directory / 'metrics.csv').open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    (directory / 'errors.json').write_text(json.dumps(errors, ensure_ascii=False, indent=2), encoding='utf-8')
    result_summary = {'texts': len(cases), 'macro_f1': macro, 'weighted_f1': weighted,
                      'corpus_ok': len(cases) >= 100 and all(counts[k] >= 50 for k in TYPES),
                      'f1_ok': macro >= .80 and weighted >= .85}
    (directory / 'summary.json').write_text(json.dumps(result_summary, ensure_ascii=False, indent=2), encoding='utf-8')
    return rows, result_summary


#===== Генерация валидного номера ИНН =====
def create_inn(prefix, rng, length=10):
    if length not in (10, 12) or len(prefix) != 4 or not prefix.isdigit():
        raise ValueError('ИНН: требуется 4-значный код органа и длина 10 или 12')
    s = prefix + ''.join(str(rng.randrange(10)) for _ in range(length - 5 if length == 10 else length - 6))
    if length == 10:
        check = sum(int(a) * b for a, b in zip(s, (2, 4, 10, 3, 5, 9, 4, 6, 8))) % 11 % 10
        return s + str(check)
    check = sum(int(a) * b for a, b in zip(s, (7, 2, 4, 10, 3, 5, 9, 4, 6, 8))) % 11 % 10
    s += str(check)
    check = sum(int(a) * b for a, b in zip(s, (3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8))) % 11 % 10
    return s + str(check)

#===== Генерация корректного номера карты =====
def create_card(prefix, length, rng):
    s = prefix + ''.join(str(rng.randrange(10)) for _ in range(length - len(prefix) - 1))
    return next(s + str(d) for d in range(10) if valid_card(s + str(d)))


#===== Генерация корректного СНИЛС =====
def create_snils(rng):
    while True:
        s = str(rng.randrange(1001999, 1000000000)).zfill(9)
        checksum = sum(int(v) * (9-i) for i, v in enumerate(s))
        checksum = checksum if checksum < 100 else (0 if checksum in (100, 101) else checksum % 101)
        checksum = 0 if checksum == 100 else checksum
        value = s + str(checksum).zfill(2)
        if valid_snils(value):
            return value


#===== Сохранение разделителей исходного идентификатора =====
def same_format(original, digits):
    it = iter(digits)
    return ''.join(next(it) if c.isdigit() else c for c in original)


#===== Морфологический разбор ФИО с выбором специального словаря =====
def person_parse(value, kind, context='', gender=None):
    import pymorphy3
    morph = person_parse._morph if hasattr(person_parse, '_morph') else pymorphy3.MorphAnalyzer()
    person_parse._morph = morph
    tag = 'Name' if kind == 'NAME' else 'Surn'
    analyses = [p for p in morph.parse(value) if tag in p.tag]
    if not analyses:
        return morph, morph.parse(value)[0]
    case = context_case(context)
    analyses.sort(key=lambda p: (
        gender is not None and p.tag.gender not in (gender, None),
        case is not None and p.tag.case != case,
        -p.score
    ))
    return morph, analyses[0]


#===== Разрешение неоднозначного падежа по предлогу =====
def context_case(context):
    # В частности «к Ивановой» (дат.) и «от Ивановой» (род.) без контекста неразличимы.
    words = re.findall(r'[А-Яа-яЁё]+', context.lower())[-3:]
    for word in reversed(words):
        if word in ('от', 'для', 'без', 'у', 'из', 'около', 'после'):
            return 'gent'
        if word in ('к', 'ко', 'благодаря'):
            return 'datv'
        if word in ('с', 'со', 'над', 'под', 'перед', 'между'):
            return 'ablt'
        if word in ('о', 'об', 'обо', 'при'):
            return 'loct'
    return None


#===== Определение рода с учётом морфологии и обращения =====
def guess_gender(value, kind, text='', start=0):
    context = text[max(0, start - 80):start].lower()
    feminine = r'\b(?:гражданка|женщина|госпожа|сотрудница|девушка|она)\b'
    masculine = r'\b(?:гражданин|мужчина|господин|сотрудник|юноша|он)\b'
    matches = list(re.finditer(feminine + '|' + masculine, context))
    if matches:
        return 'femn' if re.fullmatch(feminine, matches[-1].group()) else 'masc'
    lower = value.casefold()
    if kind == 'SURNAME':
        if lower.endswith(('овой', 'евой', 'иной', 'овой', 'овою', 'евою', 'иною',
                           'ова', 'ева', 'ина', 'ская', 'цкая', 'скую', 'цкую')):
            return 'femn'
        if lower.endswith(('овым', 'евым', 'иным', 'ова', 'ева')) and context_case(context) == 'ablt':
            return 'masc'
    _, parsed = person_parse(value, kind, context)
    if parsed.tag.gender in ('masc', 'femn'):
        return parsed.tag.gender
    lemma = parsed.normal_form.casefold()
    if lemma in ('саша', 'женя', 'валя'):
        return 'masc'
    if kind == 'NAME':
        return 'femn' if lemma.endswith(('а', 'я')) and lemma not in ('илья', 'никита', 'кузьма') else 'masc'
    return 'femn' if lemma.endswith(('ова', 'ева', 'ина', 'ская', 'цкая')) else 'masc'


#===== Замена склоняемого имени или фамилии =====
def replace_person(kind, value, text, start, rng, chosen=None, gender=None):
    context = text[max(0, start - 80):start]
    gender = gender or guess_gender(value, kind, text, start)
    morph, parsed = person_parse(value, kind, context, gender)
    names = ((FEMALE_NAMES if gender == 'femn' else MALE_NAMES) if kind == 'NAME'
             else (FEMALE_SURNAMES if gender == 'femn' else MALE_SURNAMES))
    tag = 'Name' if kind == 'NAME' else 'Surn'
    source_lemmas = {p.normal_form.casefold() for p in morph.parse(value)
                     if tag in p.tag and p.tag.gender in (gender, None)}
    source_lemmas.add(value.casefold())
    options = [w for w in names if w.casefold() not in source_lemmas]
    if chosen is not None:
        if chosen.casefold() in source_lemmas:
            raise ValueError('Замена совпадает с исходной леммой: ' + value)
        options = [chosen]
    if not options:
        raise ValueError('Нет отличающихся замен для ' + value)
    case = context_case(context) or parsed.tag.case or 'nomn'
    for candidate in (options if chosen is not None else rng.sample(options, len(options))):
        analyses = [p for p in morph.parse(candidate)
                    if tag in p.tag and p.tag.gender in (gender, None)
                    and p.tag.case == 'nomn']
        if not analyses:
            continue
        grammar = {case}
        if parsed.tag.number == 'plur':
            grammar.add('plur')
        inflected = analyses[0].inflect(grammar)
        if inflected is None or inflected.word.casefold() == value.casefold():
            continue
        word = inflected.word
        return word.upper() if value.isupper() else word.capitalize() if value.istitle() else word
    raise ValueError('Не удалось подобрать отличающуюся форму для ' + value)


#===== Генерация отличающегося валидного идентификатора =====
def replace_value(kind, value, text, start, rng):
    digits = re.sub(r'\D', '', value)
    if kind in ('NAME', 'SURNAME'):
        return replace_person(kind, value, text, start, rng)
    for _ in range(1000):
        if kind == 'INN':
            checks = digits[-(1 if len(digits) == 10 else 2):]
            candidate = create_inn(digits[:4], rng, len(digits))
            ok = candidate.endswith(checks)
            result = same_format(value, candidate)
        elif kind == 'CARD':
            result = same_format(value, create_card(digits[:6], len(digits), rng))
            ok = True
        elif kind == 'SNILS':
            result = same_format(value, create_snils(rng))
            ok = True
        elif kind == 'PASSPORT':
            result = same_format(value, digits[:4] + ''.join(str(rng.randrange(10)) for _ in range(6)))
            ok = True
        elif kind == 'PHONE':
            result = same_format(value, digits[:4] + ''.join(str(rng.randrange(10)) for _ in range(len(digits)-4)))
            ok = True
        elif kind == 'MARRIAGE':
            result = value[:-6] + ''.join(str(rng.randrange(10)) for _ in range(6))
            ok = True
        elif kind == 'CADASTRAL':
            sections = value.split(':')
            section = sections[-1]
            result = ':'.join(sections[:3] + [str(rng.randrange(1, 10**len(section))).zfill(len(section))])
            ok = True
        elif kind == 'URL':
            host = re.match(r'(https?://)?([^/]+)(/.*)?', value, re.I)
            prefix, hostname, path = host.groups()
            tld = hostname.rsplit('.', 1)[-1] if '.' in hostname else 'org'
            result = (prefix or '') + 'example' + str(rng.randrange(100000, 999999)) + '.' + tld + (path or '')
            ok = True
        else:
            raise ValueError('Неизвестный класс: ' + kind)
        if ok and result.casefold() != value.casefold() and valid_value(kind, result):
            return result
    raise RuntimeError('Не удалось создать новую корректную замену для ' + kind)


#===== Обезличивание с согласованными заменами =====
def anonymize(text, ner=None, mapping=None, rng=None):
    """Один mapping на серию текстов; без него каждый вызов генерирует новые значения."""
    rng = rng if rng is not None else secrets.SystemRandom()
    mapping = mapping if mapping is not None else {}
    entities = recognize_pii(text, ner)
    parts, end = [], len(text)
    for kind, start, stop in reversed(entities):
        value = text[start:stop]
        if kind in ('NAME', 'SURNAME'):
            gender = guess_gender(value, kind, text, start)
            _, parsed = person_parse(value, kind, text[max(0, start - 80):start], gender)
            key = (kind, parsed.normal_form.casefold(), gender)
            if key not in mapping:
                names = ((FEMALE_NAMES if gender == 'femn' else MALE_NAMES) if kind == 'NAME'
                         else (FEMALE_SURNAMES if gender == 'femn' else MALE_SURNAMES))
                tag = 'Name' if kind == 'NAME' else 'Surn'
                morph, _ = person_parse(value, kind)
                lemmas = {p.normal_form.casefold() for p in morph.parse(value)
                          if tag in p.tag and p.tag.gender in (gender, None)}
                lemmas.add(value.casefold())
                choices = [n for n in names if n.casefold() not in lemmas]
                if not choices:
                    raise ValueError('Нет доступной замены для ' + value)
                mapping[key] = rng.choice(choices)
            replacement = replace_person(kind, value, text, start, rng, mapping[key], gender)
        else:
            key = (kind, re.sub(r'\D', '', value) if kind in ('INN', 'PASSPORT', 'CARD', 'SNILS', 'PHONE') else value.casefold())
            if key not in mapping:
                mapping[key] = replace_value(kind, value, text, start, rng)
            replacement = mapping[key]
            if kind in ('INN', 'PASSPORT', 'CARD', 'SNILS', 'PHONE'):
                replacement = same_format(value, re.sub(r'\D', '', replacement))
        if replacement.casefold() == value.casefold():
            raise RuntimeError('Замена совпала с оригиналом: ' + kind)
        parts += [text[stop:end], replacement]
        end = start
    return text[:end] + ''.join(reversed(parts)), entities

#===== Маскирование для сравнения по BERTScore =====
def mask(text, entities):
    for _, start, stop in sorted(entities, key=lambda e: e[1], reverse=True):
        text = text[:start] + '*' * (stop-start) + text[stop:]
    return text


#===== Сравнение семантической близости с маскированием =====
def compare_bertscore(texts, ner, directory='kr_results'):
    try:
        from bert_score import score
    except ImportError as exc:
        raise RuntimeError('Установите bert-score') from exc
    substitutions, masked = [], []
    mapping = {}
    for text in texts:
        substituted, entities = anonymize(text, ner, mapping)
        substitutions.append(substituted)
        masked.append(mask(text, entities))
    scores = {}
    for label, samples in [('synonym', substitutions), ('mask', masked)]:
        _, _, f1 = score(samples, texts, lang='ru', verbose=False)
        scores[label] = float(f1.mean())
    directory = Path(directory)
    directory.mkdir(exist_ok=True, parents=True)
    (directory / 'bertscore.json').write_text(json.dumps(scores, ensure_ascii=False, indent=2), encoding='utf-8')
    return scores


In [3]:
#===== Блок 3. Подготовка конфигурации RuBERT (без обучения) =====
import os
import torch

MODEL_NAME = 'DeepPavlov/rubert-base-cased'
LABELS = ['O', 'B-NAME', 'I-NAME', 'B-SURNAME', 'I-SURNAME']
id2label = dict(enumerate(LABELS))
label2id = {v: k for k, v in id2label.items()}
# Здесь нет загрузки FactRuEval или обучения: они уже выполнялись ранее.
print('Конфигурация распознавания готова. Повторное обучение отключено.')


Конфигурация распознавания готова. Повторное обучение отключено.


In [ ]:
#===== Блок 4. Загрузка ранее обученной RuBERT (БЕЗ переобучения) =====
from pathlib import Path
from transformers import AutoModelForTokenClassification, AutoTokenizer

# При необходимости укажи здесь путь к СУЩЕСТВУЮЩЕЙ обученной модели.
MODEL_DIR = None
CANDIDATES = [
    '/content/КР_ПДн/rubert_model',
    '/content/rubert_model',
    './rubert_model',
    '/content/drive/MyDrive/PII_KR/rubert_model',
    '/content/kr_rubert_model',
    '/content/kr_rubert_model/checkpoint-final',
]

def has_weights(folder):
    path = Path(folder)
    return (path / 'config.json').is_file() and any(
        (path / filename).is_file() for filename in ('model.safetensors', 'pytorch_model.bin')
    )

if not any(has_weights(p) for p in CANDIDATES) and not (MODEL_DIR and has_weights(MODEL_DIR)):
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except (ImportError, ValueError, RuntimeError) as exc:
        print('Подключение Google Drive не удалось:', exc)

MODEL_DIR = MODEL_DIR or next((p for p in CANDIDATES if has_weights(p)), None)
if not MODEL_DIR or not has_weights(MODEL_DIR):
    raise FileNotFoundError(
        'Обученная RuBERT не найдена. Подключи Google Drive или укажи '
        'MODEL_DIR с config.json и model.safetensors. Обучение НЕ запускается.'
    )

print('Загружаем готовую модель:', MODEL_DIR)
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR if (Path(MODEL_DIR) / 'tokenizer_config.json').exists()
                                               else MODEL_NAME, use_fast=True)
labels_from_model = {int(k): v for k, v in model.config.id2label.items()}
if labels_from_model != id2label:
    raise ValueError('Метки сохранённой модели отличаются от NAME/SURNAME BIO: ' + str(labels_from_model))
model = model.to('cuda' if torch.cuda.is_available() else 'cpu').eval()
print('Модель загружена, повторного обучения не было.')


In [5]:
#===== Блок 5. Прогноз NAME/SURNAME с сохранением позиций в тексте =====
device = next(model.parameters()).device

def recognize_names(text):
    # Перекрывающиеся окна нужны для длинных документов, чтобы не терять сущности после 512 токенов.
    encoded = tokenizer(text, return_offsets_mapping=True,
                        return_overflowing_tokens=True, truncation=True,
                        max_length=384, stride=60, padding=False)
    best = {}
    model.eval()
    with torch.no_grad():
        for i, offsets in enumerate(encoded['offset_mapping']):
            inputs = {k: torch.tensor([encoded[k][i]], device=device)
                      for k in ('input_ids','attention_mask') if k in encoded}
            logits = model(**inputs).logits[0]
            probs = logits.softmax(-1)
            ids = probs.argmax(-1).tolist()
            words = encoded.word_ids(batch_index=i)
            previous = None
            for j, ((start, stop), word) in enumerate(zip(offsets, words)):
                if word is None or word == previous:
                    previous = word
                    continue
                previous = word
                label = id2label[ids[j]]
                confidence = float(probs[j, ids[j]])
                if (start,stop) not in best or best[(start,stop)][1] < confidence:
                    best[(start,stop)] = (label, confidence)
    spans, active = [], None
    for (start,stop), (tag,score) in sorted(best.items()):
        kind = tag[2:] if '-' in tag else None
        if active and (kind != active[0] or tag.startswith('B-') or
                       text[active[2]:start].strip()):
            spans.append(active)
            active = None
        if kind in ('NAME','SURNAME'):
            if active is None:
                active = [kind,start,stop]
            else:
                active[2] = stop
    if active:
        spans.append(active)
    return spans

example = 'Иван Петров направил паспорт: 4510 123456 и ИНН: 7707083893.'
print('Распознанные ПДн:', [(k, example[s:e]) for k,s,e in recognize_pii(example, recognize_names)])
#===== Кэширование предсказаний для повторных оценок без дополнительного инференса =====
from functools import lru_cache
recognize_names = lru_cache(maxsize=512)(recognize_names)


Распознанные ПДн: [('NAME', 'Иван'), ('SURNAME', 'Петров'), ('PASSPORT', '4510 123456'), ('INN', '7707083893')]


In [6]:
#===== Блок 6. Сложный контрольный корпус и две оценки F1 =====
# Данные НЕ размечаются предсказаниями программы.
# Это полусинтетическая проверка, а не независимая ручная разметка документов.
"""Оценка КР: отдельный сложный корпус, entity-level и token-level BIO.

Корпус полусинтетический, создан независимо от предсказаний детектора.
Он НЕ заменяет оценку на вручную размеченных реальных документах.
"""
import csv
import json
import random
import re
from collections import Counter
from pathlib import Path



FIRST = ('Константин', 'Виктор', 'Роман', 'Егор', 'Арсений', 'Борис', 'Максим',
         'Артём', 'Глеб', 'Владислав', 'Вероника', 'Алиса', 'Дарья', 'Полина',
         'Ксения', 'Лидия', 'Валерия', 'Юлия', 'Надежда', 'Анастасия')
LAST = ('Морозов', 'Белов', 'Зайцев', 'Крылов', 'Богданов', 'Воробьёв', 'Фролов',
        'Громов', 'Денисов', 'Тихонов', 'Морозова', 'Белова', 'Зайцева', 'Крылова',
        'Богданова', 'Воробьёва', 'Фролова', 'Громова', 'Денисова', 'Тихонова')

# Варианты контекста подобраны отдельно от шаблонов распознавателя.
PHRASES = {
    'NAME': ('В обращении фигурирует {v}.', 'С заявлением выступил {v}.',
             'Имя заявителя — {v}.', 'Заявитель представился: {v}.',
             'Указано имя: {v}.', 'На приём пришёл {v}.'),
    'SURNAME': ('Документ подписал гражданин {v}.', 'Сведения о заявителе {v} переданы.',
                'Фамилия — {v}.', 'Подпись заявителя: {v}.',
                'В протоколе отмечен {v}.', 'Указана фамилия: {v}.'),
    'INN': ('Налоговый идентификатор заявителя {v}.', 'Реквизиты организации: ИНН {v}.',
            'В сведениях указан ИНН: {v}.', 'Налогоплательщик сообщил ИНН № {v}.',
            'Проверьте идентификационный номер налогоплательщика {v}.',
            'Для сверки записали ИНН — {v}.'),
    'PASSPORT': ('Паспортные сведения: {v}.', 'Для пропуска предъявлен паспорт {v}.',
                 'Документ удостоверения личности: {v}.', 'Номер паспорта {v}.',
                 'В заявлении указана серия паспорта {v}.', 'Проверили паспорт РФ: {v}.'),
    'CARD': ('Для возврата указали карту {v}.', 'Платёжный реквизит: {v}.',
             'Банковская карта заявителя — {v}.', 'Оплата проведена с карты {v}.',
             'Номер карты: {v}.', 'Получатель указал номер карты {v}.'),
    'SNILS': ('В анкете записан СНИЛС {v}.', 'СНИЛС застрахованного лица: {v}.',
              'Лицевой счёт в ПФР — {v}.', 'Проверка СНИЛС № {v}.',
              'Страховой номер: {v}.', 'В личном деле СНИЛС {v}.'),
    'URL': ('Подробнее по адресу {v}.', 'Открыть страницу {v}.',
            'Ссылка на ресурс: {v}.', 'Информация опубликована на {v}.',
            'Веб-сайт проекта — {v}.', 'Портал доступен: {v}.'),
    'PHONE': ('Связь по номеру {v}.', 'Телефон для обратной связи: {v}.',
              'Наберите {v} после обеда.', 'Контактный номер {v}.',
              'Для уточнения звоните {v}.', 'Оставили телефон: {v}.'),
    'MARRIAGE': ('Актовая запись о браке: {v}.', 'Свидетельство о браке {v}.',
                 'Подтверждение регистрации брака — {v}.', 'Серия свидетельства о браке: {v}.',
                 'Документ о заключении брака № {v}.', 'Реквизиты акта о браке: {v}.'),
    'CADASTRAL': ('Участок обозначен номером {v}.', 'Кадастровый номер: {v}.',
                  'В выписке указан объект {v}.', 'Объект недвижимости {v}.',
                  'Росреестр: кадастровый номер {v}.', 'Идентификатор участка: {v}.'),
}
NEGATIVE = (
    'Договор № 4510 123456 оформлен без паспортных данных.',
    'Задача 12:34:5678901:123 из примера не относится к реестру недвижимости.',
    'В учебном тексте обсуждается слово Иван как существительное.',
    'Сумма по счёту составила 4111 1111 1111 1111 условных единиц.',
    'Ключ для тестовой среды: 7707083893; это не ИНН клиента.',
    'Файл сохранён под названием III-АБ №123456, не свидетельство о браке.',
    'Номер заявления 112-233-445 95 не является номером СНИЛС.',
    'Внутренний инвентарный код +7 (999) 123-45-67 используется как образец.',
    'Адрес почты test@example.org в шаблоне не является URL-полем.',
    'Обсуждались обычные слова: Роман, Надежда, Морозов.',
    'Код 1234567890 имеет неверный контрольный разряд.',
    'Строка 1234 5678 9012 3456 не является платёжной картой.',
)

#===== Создание примера с независимыми символьными границами =====
def annotated_text(fragments):
    text, spans = '', []
    for kind, fragment, value in fragments:
        before, after = fragment.split('{v}')
        text += before
        start = len(text)
        text += value
        if kind:
            spans.append([kind, start, len(text)])
        text += after + ' '
    return {'text': text.strip(), 'spans': spans}


#===== Генерация разнообразных синтетических значений =====
def value_for(kind, i, rng):
    if kind == 'NAME':
        return FIRST[(i * 7 + 3) % len(FIRST)]
    if kind == 'SURNAME':
        return LAST[(i * 9 + 5) % len(LAST)]
    if kind == 'INN':
        return create_inn('%04d' % rng.randrange(1000, 9999), rng, 10 if i % 2 else 12)
    if kind == 'PASSPORT':
        return '%04d %06d' % (rng.randrange(1000, 9999), rng.randrange(100000, 999999))
    if kind == 'CARD':
        number = create_card('427638', 16, rng)
        return number if i % 3 else ' '.join(number[k:k+4] for k in range(0, 16, 4))
    if kind == 'SNILS':
        number = create_snils(rng)
        return number[:3] + '-' + number[3:6] + '-' + number[6:9] + ' ' + number[9:]
    if kind == 'URL':
        return 'https://portal-%d.example.org/req/%d' % (i, rng.randrange(10000))
    if kind == 'PHONE':
        return '+7 (9%02d) %03d-%02d-%02d' % (rng.randrange(100), rng.randrange(1000),
                                              rng.randrange(100), rng.randrange(100))
    if kind == 'MARRIAGE':
        return rng.choice(('II-АБ', 'III-ВГ', 'IV-ДЕ')) + ' №%06d' % rng.randrange(100000, 999999)
    if kind == 'CADASTRAL':
        return '%02d:%02d:%07d:%d' % (rng.randrange(1, 90), rng.randrange(1, 90),
                                      rng.randrange(1000000, 9999999), rng.randrange(1, 9999))
    raise ValueError(kind)


#===== Создание диагностического корпуса без использования предсказаний =====
def build_challenge(seed=20261120):
    rng = random.Random(seed)
    cases = []
    # 120 документов, в каждом пять разных атрибутов; по 60 каждого вида.
    for i in range(120):
        kinds = [TYPES[(i + j) % len(TYPES)] for j in range(5)]
        rng.shuffle(kinds)
        parts = []
        for j, kind in enumerate(kinds):
            formats = PHRASES[kind]
            variant = (i // 5 + j * 3) % len(formats)
            parts.append((kind, formats[variant], value_for(kind, i * 5 + j, rng)))
        cases.append(annotated_text(parts))
    # 40 отрицательных текстов. Разметка не строится из результатов программы.
    cases.extend({'text': NEGATIVE[i % len(NEGATIVE)], 'spans': []} for i in range(40))
    expected = Counter(kind for case in cases for kind, _, _ in case['spans'])
    assert len(cases) >= 100 and all(expected[k] >= 50 for k in TYPES), expected
    for case in cases:
        for kind, begin, end in case['spans']:
            assert kind in TYPES and 0 <= begin < end <= len(case['text'])
    return cases


#===== Расчёт точности, полноты и F1 =====
def counts_to_row(kind, c):
    tp, fp, fn = c['tp'], c['fp'], c['fn']
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    return {'attribute': kind, 'precision': p, 'recall': r, 'f1': f1,
            'support': tp + fn, 'tp': tp, 'fp': fp, 'fn': fn}


#===== Оценка полного совпадения границ и типов сущностей =====
def evaluate_entity(cases, ner):
    totals = {kind: Counter() for kind in TYPES}
    errors = []
    for i, case in enumerate(cases):
        gold = set(map(tuple, case['spans']))
        predicted = set(map(tuple, recognize_pii(case['text'], ner)))
        for kind in TYPES:
            g = {s for s in gold if s[0] == kind}
            p = {s for s in predicted if s[0] == kind}
            totals[kind].update(tp=len(g & p), fp=len(p-g), fn=len(g-p))
        if gold != predicted:
            errors.append({'case': i, 'text': case['text'],
                           'expected': sorted(gold), 'predicted': sorted(predicted)})
    return [counts_to_row(k, totals[k]) for k in TYPES], errors


#===== Оценка токенной BIO-классификации без фоновой метки O =====
def evaluate_token(cases, ner):
    totals = {kind: Counter() for kind in TYPES}
    for case in cases:
        gold = spans_to_bio(case['text'], case['spans'])
        predicted = spans_to_bio(case['text'], recognize_pii(case['text'], ner))
        assert len(gold) == len(predicted) == len(TOKEN.findall(case['text']))
        for actual, guess in zip(gold, predicted):
            for kind in TYPES:
                # Классификация по типу сущности, сохраняя значение B/I.
                relevant = ('B-' + kind, 'I-' + kind)
                if actual == guess and actual in relevant:
                    totals[kind]['tp'] += 1
                else:
                    if guess in relevant:
                        totals[kind]['fp'] += 1
                    if actual in relevant:
                        totals[kind]['fn'] += 1
    return [counts_to_row(k, totals[k]) for k in TYPES]


#===== Усреднение метрик по десяти классам =====
def summarize(rows, text_count):
    support = sum(row['support'] for row in rows)
    macro = sum(row['f1'] for row in rows) / len(rows)
    weighted = sum(row['support'] * row['f1'] for row in rows) / support if support else 0
    return {'texts': text_count, 'support': support, 'macro_f1': macro,
            'weighted_f1': weighted, 'passes_thresholds': macro >= .80 and weighted >= .85}


#===== Сохранение данных и метрик для воспроизведения =====
def run_evaluation(cases, ner, destination):
    path = Path(destination)
    path.mkdir(parents=True, exist_ok=True)
    entity, errors = evaluate_entity(cases, ner)
    token = evaluate_token(cases, ner)
    for filename, rows in [('entity_metrics.csv', entity), ('token_metrics.csv', token)]:
        with (path / filename).open('w', encoding='utf-8-sig', newline='') as output:
            writer = csv.DictWriter(output, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
    summaries = {'entity': summarize(entity, len(cases)),
                 'token_bio': summarize(token, len(cases)),
                 'note': 'Полусинтетический тест. Не выдавать за независимый реальный корпус.'}
    for filename, content in [('test_cases.json', cases), ('errors.json', errors),
                              ('summary.json', summaries)]:
        (path / filename).write_text(json.dumps(content, ensure_ascii=False, indent=2),
                                     encoding='utf-8')
    return entity, token, summaries, errors

challenge_cases = build_challenge()
entity_rows, token_rows, summaries, detection_errors = run_evaluation(
    challenge_cases, recognize_names, '/content/kr_results/challenge'
)
print('Количество текстов:', len(challenge_cases))
for kind, rows in [('ENTITY-LEVEL', entity_rows), ('TOKEN BIO', token_rows)]:
    print('\n' + kind)
    for row in rows:
        print('%-10s P=%.4f R=%.4f F1=%.4f N=%d' %
              (row['attribute'], row['precision'], row['recall'], row['f1'], row['support']))
    key = 'entity' if kind == 'ENTITY-LEVEL' else 'token_bio'
    print('ИТОГО:', summaries[key])
print('Текстов с ошибками:', len(detection_errors))
print('Файлы: /content/kr_results/challenge')


#===== Дополнительная регрессионная проверка вне генератора корпуса =====
regression_cases = [
    annotated_text([('INN', 'Налоговый идентификатор заявителя {v}.', '7707083893')]),
    annotated_text([('PASSPORT', 'В анкете оставлены паспортные сведения: {v}.', '4510 123456')]),
    annotated_text([('MARRIAGE', 'Актовая запись о браке: {v}.', 'II-АБ №123456')]),
    annotated_text([('NAME', 'Имя заявителя — {v}.', 'Константин')]),
    annotated_text([('SURNAME', 'Подпись заявителя: {v}.', 'Громова')]),
    {'text': 'Общая сумма составила 15000 рублей.', 'spans': []},
]
reg_rows, reg_errors = evaluate_entity(regression_cases, recognize_names)
print('Регрессионные случаи:', len(regression_cases),
      'ошибок:', len(reg_errors), '(не используются для итоговой оценки)')
for err in reg_errors:
    print('Регрессионная ошибка:', err)


Количество текстов: 160

ENTITY-LEVEL
SURNAME    P=1.0000 R=0.6667 F1=0.8000 N=60
NAME       P=1.0000 R=0.4667 F1=0.6364 N=60
INN        P=1.0000 R=1.0000 F1=1.0000 N=60
PASSPORT   P=1.0000 R=1.0000 F1=1.0000 N=60
CARD       P=0.9375 R=1.0000 F1=0.9677 N=60
SNILS      P=0.9524 R=1.0000 F1=0.9756 N=60
URL        P=1.0000 R=1.0000 F1=1.0000 N=60
PHONE      P=0.9524 R=1.0000 F1=0.9756 N=60
MARRIAGE   P=1.0000 R=1.0000 F1=1.0000 N=60
CADASTRAL  P=0.9375 R=1.0000 F1=0.9677 N=60
ИТОГО: {'texts': 160, 'support': 600, 'macro_f1': 0.93230670195265, 'weighted_f1': 0.93230670195265, 'passes_thresholds': True}

TOKEN BIO
SURNAME    P=1.0000 R=0.6667 F1=0.8000 N=60
NAME       P=1.0000 R=0.4667 F1=0.6364 N=60
INN        P=1.0000 R=1.0000 F1=1.0000 N=60
PASSPORT   P=1.0000 R=1.0000 F1=1.0000 N=120
CARD       P=0.8609 R=1.0000 F1=0.9252 N=99
SNILS      P=0.9524 R=1.0000 F1=0.9756 N=360
URL        P=1.0000 R=1.0000 F1=1.0000 N=900
PHONE      P=0.9524 R=1.0000 F1=0.9756 N=600
MARRIAGE   P=1.0000 R=1.000

In [7]:
#===== Блок 7. Проверка синонимического обезличивания =====
from collections import Counter

# Проверяем несколько вариантов текста. Пропуски распознавания НЕ скрываем.
examples = [challenge_cases[i] for i in (0, 1, 2, 5, 10)]
shared_mapping = {}
for case in examples:
    result, detected = anonymize(case['text'], recognize_names, shared_mapping)
    again, _ = anonymize(case['text'], recognize_names, shared_mapping)
    assert result == again, 'Нарушено постоянство замен внутри запуска'
    for kind, begin, end in detected:
        old = case['text'][begin:end]
        # Исходная сущность может встречаться в другой части текста как обычное слово.
        assert result != case['text'], 'Текст не изменился после обработки'
    print('До:', case['text'][:130])
    print('После:', result[:130])
    print('Найдено:', len(detected), '/', len(case['spans']), 'эталонных сущностей')

# Метрика покрытия обезличиванием: сколько эталонных сущностей найдено точно.
true_positives = sum(len(set(map(tuple, case['spans'])) &
                         set(map(tuple, recognize_pii(case['text'], recognize_names))))
                     for case in challenge_cases)
gold_total = sum(len(case['spans']) for case in challenge_cases)
print('Доля эталонных сущностей, обнаруженных для замены:',
      '%.4f' % (true_positives / gold_total))
print('Пропущено:', gold_total - true_positives, 'из', gold_total)


До: Налоговый идентификатор заявителя 327861622499. Подпись заявителя: Богданова. В обращении фигурирует Юлия. Оплата проведена с карт
После: Налоговый идентификатор заявителя 327814956399. Подпись заявителя: Иванова. В обращении фигурирует Юлия. Оплата проведена с карты 
Найдено: 4 / 5 эталонных сущностей
До: В анкете записан СНИЛС 396-693-945 72. Налогоплательщик сообщил ИНН № 615765508914. Для возврата указали карту 4276383536706509. Н
После: В анкете записан СНИЛС 585-055-729 21. Налогоплательщик сообщил ИНН № 615705784814. Для возврата указали карту 4276384285213366. Н
Найдено: 4 / 5 эталонных сущностей
До: Налоговый идентификатор заявителя 727346509702. Информация опубликована на https://portal-11.example.org/req/639. Паспортные сведе
После: Налоговый идентификатор заявителя 727385185402. Информация опубликована на https://example264723.org/req/639. Паспортные сведения:
Найдено: 5 / 5 эталонных сущностей
До: СНИЛС застрахованного лица: 041-273-140 08. Веб-сайт проекта — https://p

In [8]:
#===== Блок 8. BERTScore: сравнение с маскированием =====
# BERTScore не измеряет полноту распознавания, её см. блоки 6–7.
# Ограничиваемся 40 текстами, чтобы снизить вычислительную нагрузку.
samples = [case['text'] for case in challenge_cases[:40]]
score_result = compare_bertscore(samples, recognize_names,
                                 '/content/kr_results/challenge')
print('BERTScore F1 для синтетических замен:', score_result['synonym'])
print('BERTScore F1 для маскирования:', score_result['mask'])
print('Разница:', score_result['synonym'] - score_result['mask'])
print('Внимание: сравнение только на первых 40 полусинтетических текстах.')


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

BERTScore F1 для синтетических замен: 0.9570772051811218
BERTScore F1 для маскирования: 0.7155631184577942
Разница: 0.24151408672332764
Внимание: сравнение только на первых 40 полусинтетических текстах.


In [9]:
#===== Блок 9. Проверки после изменений =====
import random
r = random.Random(19)
for _ in range(30):
    assert valid_inn(create_inn('7722', r, 12))
    assert valid_inn(create_inn('7722', r, 10))
    assert valid_card(create_card('427638', 16, r))
    assert valid_snils(create_snils(r))
assert 'INN' in [k for k, _, _ in recognize_pii('Налоговый идентификатор заявителя ' + create_inn('7722', r, 12), recognize_names)]
assert 'PASSPORT' in [k for k, _, _ in recognize_pii('Паспортные сведения: 4510 123456', recognize_names)]
assert 'MARRIAGE' in [k for k, _, _ in recognize_pii('Актовая запись о браке: II-АБ №123456', recognize_names)]
print('Регрессионные проверки успешно выполнены.')


Регрессионные проверки успешно выполнены.


## Блок 10. Регрессионная проверка после исправлений

Контрольные случаи предыдущего запуска уже изучены, поэтому результаты этого блока **не являются новой независимой оценкой**. Выполняется без обучения RuBERT. Здесь также проверяются замены идентификаторов и склонение имён и фамилий по контексту (например, «от Ивановой» и «к Ивановой»).

In [10]:
"""Дополнительная проверка КР на фиксированном контрольном корпусе.

Данные синтетические, не размечаются предсказаниями. После разбора
предыдущих ошибок они используются только для регрессионной проверки. Для полноценной внешней оценки нужен реальный размеченный корпус.
"""
import json
import random
import re
from collections import Counter
from pathlib import Path


NAMES = ('Марк', 'Тимофей', 'Платон', 'Ярослав', 'Степан', 'Филипп',
         'Лев', 'Матвей', 'Милана', 'Агата', 'София', 'Василиса',
         'Арина', 'Ульяна', 'Елизавета', 'Таисия')
SURNAMES = ('Соколов', 'Ершов', 'Лебедев', 'Быков', 'Титов', 'Королёв',
            'Гусев', 'Миронов', 'Соколова', 'Ершова', 'Лебедева', 'Быкова',
            'Титова', 'Королёва', 'Гусева', 'Миронова')
# Контексты написаны отдельно от основного корпуса и не повторяют его формулировки.
CONTEXTS = {
    'NAME': ('В графе с личными сведениями встречается {v}.',
             'Участник по имени {v} прислал дополнение.',
             'В реестре обращений записали {v}.',
             'Посетитель назвал себя {v}.',
             'В рабочей переписке упоминается {v}.'),
    'SURNAME': ('Ответственный за документ — {v}.',
                'По журналу проходил гражданин с фамилией {v}.',
                'Заявление зарегистрировано на {v}.',
                'Руководителю передали материалы от {v}.',
                'В списке участников значится {v}.'),
    'INN': ('В карточке контрагента идентификатор налогоплательщика {v}.',
            'В налоговой декларации стоит номер {v}.',
            'Рядом с реквизитами налоговый номер {v}.',
            'Для сверки передали значение ИНН {v}.',
            'У сотрудника запросили ИНН {v}.'),
    'PASSPORT': ('При оформлении доверенности указан паспортный номер {v}.',
                 'Предъявленный удостоверяющий документ имеет реквизиты {v}.',
                 'В анкете отметили паспорт серия/номер {v}.',
                 'Секретарь записал паспорт {v}.',
                 'Паспортные данные заявителя — {v}.'),
    'CARD': ('Реквизиты пластиковой карты указаны как {v}.',
             'В платёжном поручении номер карты {v}.',
             'Карту для компенсации записали: {v}.',
             'Счёт пополнили с банковской карты {v}.',
             'Для перечисления указана карта {v}.'),
    'SNILS': ('В пенсионной анкете страховой номер равен {v}.',
              'Полученный от заявителя СНИЛС {v}.',
              'Для формирования личного дела внесли СНИЛС {v}.',
              'Проверяем страховой номер индивидуального лицевого счёта {v}.',
              'В переданной форме номер СНИЛС {v}.'),
    'URL': ('Сведения опубликованы по ссылке {v}.',
            'Инструкция выложена здесь: {v}.',
            'Для загрузки отчёта указан веб-адрес {v}.',
            'Электронный ресурс находится на {v}.',
            'Публичная страница доступна по {v}.'),
    'PHONE': ('Для звонка оставили контакт {v}.',
              'Оператору сообщили мобильный {v}.',
              'В поле связи проставлен номер {v}.',
              'Диспетчер набрал телефон {v}.',
              'Обратная связь предусмотрена по {v}.'),
    'MARRIAGE': ('В архиве хранится брачное свидетельство серии {v}.',
                 'В выписку перенесли реквизиты свидетельства о заключении брака {v}.',
                 'Заявитель показал свидетельство о браке {v}.',
                 'В списке актов гражданского состояния записано свидетельство о браке {v}.',
                 'Данные документа о регистрации брака: {v}.'),
    'CADASTRAL': ('Для объекта указан кадастровый идентификатор {v}.',
                  'Из ЕГРН выписали номер участка {v}.',
                  'В карточке недвижимости стоит кадастровый № {v}.',
                  'Земельный участок учтён как {v}.',
                  'В договоре назван кадастровый номер {v}.'),
}

# Отрицательные случаи намеренно не содержат валидных реквизитов из положительной выборки.
NEGATIVE_BASE = (
    'Секретарь обсудил график выдачи оборудования на следующую неделю.',
    'В отчёте упоминались показатели точности и полноты классификации.',
    'Слово «роман» здесь обозначает литературный жанр.',
    'На складе хранится 17 коробок бумаги для принтеров.',
    'Группа проверяла расписание занятий и номера аудиторий.',
    'В переписке рассмотрели обычную финансовую смету.',
    'Участники согласовали состав итоговой документации.',
    'Инженер обнаружил ошибку в локальной сети лаборатории.',
    'Документ содержит условный шестизначный код 123456.',
    'Указана дата 12.05.2024 и время 14:35, без контактов.',
    'В справочнике используется запись 00-000-000-00, она некорректна.',
    'Фиктивный налоговый номер 0000000001 не является валидным ИНН.',
    'Тестовая карта 1234 1234 1234 1234 не прошла проверку Луна.',
    'В условии задачи фигурируют числа 45, 10 и 123456.',
    'Серия документа не задана: вместо неё стоит ХХ-00.',
    'Сотрудники обсудили логистику доставки коробок.',
    'В инструкции приводится абстрактная модель системы.',
    'В математическом примере вычисляют 12:34 и 56:78.',
    'База данных хранит только число выполненных задач.',
    'Подпись в шаблоне отсутствует, поля оставлены пустыми.',
)


#===== Генерация самостоятельных значений проверочных реквизитов =====
def sample_value(kind, number, rng):
    if kind == 'NAME':
        return NAMES[(number * 3 + 1) % len(NAMES)]
    if kind == 'SURNAME':
        return SURNAMES[(number * 5 + 2) % len(SURNAMES)]
    if kind == 'INN':
        return create_inn(str(rng.randrange(1000, 9999)), rng, 12 if number % 2 else 10)
    if kind == 'PASSPORT':
        return '%04d %06d' % (rng.randrange(1000, 9999), rng.randrange(100000, 1000000))
    if kind == 'CARD':
        digits = create_card('546912', 16, rng)
        return ' '.join(digits[i:i+4] for i in range(0, 16, 4)) if number % 2 else digits
    if kind == 'SNILS':
        digits = create_snils(rng)
        return '%s-%s-%s %s' % (digits[:3], digits[3:6], digits[6:9], digits[9:])
    if kind == 'URL':
        return 'https://docs-%d.example.net/archive/%d' % (number, rng.randrange(100000))
    if kind == 'PHONE':
        return '+7 (9%02d) %03d-%02d-%02d' % (rng.randrange(100), rng.randrange(1000),
                                               rng.randrange(100), rng.randrange(100))
    if kind == 'MARRIAGE':
        return rng.choice(('V-ЖЗ', 'VI-ИК', 'VII-ЛМ')) + ' №%06d' % rng.randrange(100000, 1000000)
    if kind == 'CADASTRAL':
        return '%02d:%02d:%07d:%d' % (rng.randrange(1, 90), rng.randrange(1, 90),
                                      rng.randrange(1000000, 10000000), rng.randrange(1, 99999))
    raise ValueError(kind)


#===== Формирование фиксированного контрольного корпуса с эталонными границами =====
def build_holdout(seed=971204):
    rng, cases = random.Random(seed), []
    for i in range(120):
        selected = [TYPES[(i + j) % 10] for j in range(5)]
        rng.shuffle(selected)
        parts, positions, offset = [], [], 0
        for j, kind in enumerate(selected):
            phrase = CONTEXTS[kind][(i // 10 + j) % 5]
            value = sample_value(kind, i * 5 + j, rng)
            before, after = phrase.split('{v}')
            piece = before + value + after
            start = offset + len(before)
            positions.append([kind, start, start + len(value)])
            parts.append(piece)
            offset += len(piece) + 1
        cases.append({'text': ' '.join(parts), 'spans': positions})
    for i in range(40):
        cases.append({'text': NEGATIVE_BASE[i % len(NEGATIVE_BASE)] +
                              ' Номер фрагмента описания: %d.' % (i + 1), 'spans': []})
    counts = Counter(span[0] for case in cases for span in case['spans'])
    assert len(cases) == 160 and all(counts[k] == 60 for k in TYPES), counts
    assert len({case['text'] for case in cases}) == len(cases)
    for case in cases:
        for kind, start, stop in case['spans']:
            assert 0 <= start < stop <= len(case['text'])
            assert case['text'][start:stop].strip()
    return cases


#===== Проверка формата, контрольных цифр и неповторяемости замен =====
def audit_replacements(seed=9721):
    rng = random.Random(seed)
    results = []
    for kind in TYPES:
        if kind in ('NAME', 'SURNAME'):
            continue
        for i in range(20):
            source = sample_value(kind, 1000 + i, rng)
            substitute = replace_value(kind, source, '', 0, rng)
            valid = (source != substitute and valid_value(kind, substitute))
            a, b = re.sub(r'\D', '', source), re.sub(r'\D', '', substitute)
            if kind in ('INN', 'CARD', 'PASSPORT', 'SNILS', 'PHONE'):
                valid &= len(a) == len(b)
            if kind == 'INN':
                valid &= a[:4] == b[:4] and a[-(1 if len(a) == 10 else 2):] == b[-(1 if len(b) == 10 else 2):]
                valid &= valid_inn(substitute)
            elif kind == 'PASSPORT':
                valid &= a[:4] == b[:4]
            elif kind == 'CARD':
                valid &= valid_card(substitute)
            elif kind == 'SNILS':
                valid &= valid_snils(substitute)
            elif kind == 'PHONE':
                valid &= a[:4] == b[:4]
            elif kind == 'CADASTRAL':
                valid &= source.split(':')[:3] == substitute.split(':')[:3]
            elif kind == 'MARRIAGE':
                valid &= source[:-6] == substitute[:-6]
            results.append({'type': kind, 'source': source, 'replacement': substitute, 'ok': bool(valid)})
    return results


#===== Проверка морфологии замен в разных падежах =====
def audit_morphology():
    # Падеж «Ивановой» неоднозначен: проверяем его в содержательном контексте.
    tests = [
        ('NAME', 'Ивану', 'masc', 'datv', 'Письмо к '),
        ('NAME', 'Петром', 'masc', 'ablt', 'Разговор с '),
        ('NAME', 'Анне', 'femn', 'datv', 'Документ к '),
        ('NAME', 'Марией', 'femn', 'ablt', 'Встреча с '),
        ('SURNAME', 'Ивановой', 'femn', 'gent', 'Ответ от '),
        ('SURNAME', 'Ивановой', 'femn', 'datv', 'Обращение к '),
        ('SURNAME', 'Петровым', 'masc', 'ablt', 'Встреча с '),
    ]
    results = []
    for kind, source, gender, case, context in tests:
        try:
            target = replace_person(kind, source, context + source, len(context),
                                    random.Random(19), gender=gender)
            morph, parsed = person_parse(target, kind, context, gender)
            tag = 'Name' if kind == 'NAME' else 'Surn'
            candidates = [p for p in morph.parse(target)
                          if tag in p.tag and p.tag.case == case
                          and p.tag.gender in (gender, None)]
            source_lemmas = {p.normal_form.casefold() for p in morph.parse(source)
                             if tag in p.tag and p.tag.gender in (gender, None)}
            target_lemmas = {p.normal_form.casefold() for p in candidates}
            valid = (bool(candidates) and target.casefold() != source.casefold()
                     and target_lemmas.isdisjoint(source_lemmas))
            results.append({'type': kind, 'source': source, 'context': context,
                            'replacement': target, 'expected_case': case,
                            'actual_case': candidates[0].tag.case if candidates else parsed.tag.case,
                            'ok': bool(valid)})
        except Exception as exc:
            results.append({'type': kind, 'source': source, 'expected_case': case,
                            'ok': False, 'error': str(exc)})
    return results


#===== Проверка согласованности обезличивания в рамках запуска =====
def audit_consistency():
    text = 'Имя: Иван. Имя: Иван. Фамилия: Петров. Фамилия: Петров.'
    mapping = {}
    one, _ = anonymize(text, None, mapping, random.Random(88))
    two, _ = anonymize(text, None, mapping, random.Random(99))
    return {'input': text, 'output': one, 'same_mapping_same_result': one == two,
            'changed': one != text, 'detected': recognize_pii(text)}


#===== Запуск проверки и сохранение отдельных результатов =====
def run_audit(ner, destination='/content/kr_results/holdout'):
    output = Path(destination)
    output.mkdir(parents=True, exist_ok=True)
    cases = build_holdout()
    entity, token, summary, errors = run_evaluation(cases, ner, output)
    replacements = audit_replacements()
    try:
        import pymorphy3
    except ImportError:
        morphology = [{'ok': False, 'skipped': 'pymorphy3 отсутствует'}]
        consistency = {'skipped': 'pymorphy3 отсутствует'}
    else:
        morphology = audit_morphology()
        try:
            consistency = audit_consistency()
        except Exception as exc:
            consistency = {'ok': False, 'error': str(exc)}
    audit = {'format_passed': sum(r['ok'] for r in replacements),
             'format_total': len(replacements),
             'morphology_passed': sum(r['ok'] for r in morphology),
             'morphology_total': len(morphology),
             'consistency': consistency,
             'replacement_cases': replacements,
             'morphology_cases': morphology,
             'note': ('Регрессионный синтетический набор после анализа ошибок; не является внешним '
                      'независимым корпусом реальных документов. Данные не '
                      'использовать для дальнейшей настройки и повторного отчёта как holdout.')}
    (output / 'replacement_audit.json').write_text(
        json.dumps(audit, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
    return summary, audit, len(errors)


#===== Блок 10. Запуск дополнительной контрольной оценки без обучения =====
final_summary, final_audit, final_errors = run_audit(
    recognize_names, '/content/kr_results/holdout'
)
print('Контрольный набор: 160 текстов, по 60 эталонных сущностей каждого класса.')
for title, key in [('Entity-level', 'entity'), ('Token-level BIO', 'token_bio')]:
    result = final_summary[key]
    print('%s: Macro F1 = %.4f; Weighted F1 = %.4f; пороги: %s' % (
        title, result['macro_f1'], result['weighted_f1'], result['passes_thresholds']))
print('Текстов с ошибками:', final_errors)
print('Формат и валидаторы замен: %d/%d' % (
    final_audit['format_passed'], final_audit['format_total']))
print('Падежи: %d/%d' % (final_audit['morphology_passed'], final_audit['morphology_total']))
print('Согласованность:', final_audit['consistency'])
print('Файлы контроля: /content/kr_results/holdout')


Контрольный набор: 160 текстов, по 60 эталонных сущностей каждого класса.
Entity-level: Macro F1 = 0.8864; Weighted F1 = 0.8864; пороги: True
Token-level BIO: Macro F1 = 0.8949; Weighted F1 = 0.9776; пороги: True
Текстов с ошибками: 63
Формат и валидаторы замен: 160/160
Падежи: 7/7
Согласованность: {'input': 'Имя: Иван. Имя: Иван. Фамилия: Петров. Фамилия: Петров.', 'output': 'Имя: Алексей. Имя: Алексей. Фамилия: Кузнецов. Фамилия: Кузнецов.', 'same_mapping_same_result': True, 'changed': True, 'detected': [['NAME', 5, 9], ['NAME', 16, 20], ['SURNAME', 31, 37], ['SURNAME', 48, 54]]}
Файлы контроля: /content/kr_results/holdout
